# Preprocesamiento y descriptores

**Fase:** E5

## Objetivo

Definir el ventanado, el particionado sin fuga temporal y los descriptores extraídos de cada ventana.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Fija la unidad de análisis a la que se anclará cada observación citada en una explicación.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
# `tables/` y `figures/` están fuera del control de versiones: un clon no los
# trae. `artifacts/` es su copia versionada y es la que el tribunal tiene.
ARTEFACTOS_TABLAS = RAIZ / 'artifacts' / 'tables'
ARTEFACTOS_FIGURAS = RAIZ / 'artifacts' / 'figures'

import pandas as pd

from mecanico_ia_exp.config import paths


## Estudio de longitud de ventana

El criterio no fue el número más alto, sino la **concordancia con el mecanismo**: que el efecto caiga sobre una señal que la ficha del fallo declara.

In [2]:
est = pd.read_csv(ARTEFACTOS_TABLAS / 'E5_estudio_ventana.csv')
est[['length_s', 'fault_id', 'noise_floor_p90', 'best_delta',
     'best_feature', 'feature_declared_by_fault', 'above_noise_floor']]

,length_s,fault_id,noise_floor_p90,best_delta,best_feature,feature_declared_by_fault,above_noise_floor
0,10.0,F01_vacuum_leak,0.3259,1.0000,stft_bank1_mean,True,True
1,10.0,F02_maf_drift,0.3259,0.9664,stft_bank1_mean,True,True
2,10.0,F03_thermostat_stuck,0.3259,0.8038,coolant_temperature_range,True,True
3,10.0,F04_o2_lag,0.3259,1.0000,oxygen_sensor_std,True,True
4,10.0,F05_misfire,0.3259,0.2251,coolant_temperature_min,False,False
5,30.0,F01_vacuum_leak,0.3653,1.0000,stft_bank1_mean,True,True
6,30.0,F02_maf_drift,0.3653,0.9895,stft_bank1_mean,True,True
7,30.0,F03_thermostat_stuck,0.3653,0.7285,coolant_temperature_delta_max,True,True
8,30.0,F04_o2_lag,0.3653,1.0000,oxygen_sensor_std,True,True
9,30.0,F05_misfire,0.3653,0.2705,oxygen_sensor_mean,True,False


### El suelo de ruido: por qué a 30 s el fallo de encendido era ruido

In [3]:
f05 = est[est.fault_id == 'F05_misfire'][
    ['length_s', 'noise_floor_p90', 'best_delta',
     'best_feature', 'above_noise_floor']]
f05

,length_s,noise_floor_p90,best_delta,best_feature,above_noise_floor
4,10.0,0.3259,0.2251,coolant_temperature_min,False
9,30.0,0.3653,0.2705,oxygen_sensor_mean,False
14,60.0,0.4261,0.4325,oxygen_sensor_mean,True
19,120.0,0.5133,0.6426,oxygen_sensor_mean,True


## El reparto no filtra información entre particiones

In [4]:
meta = pd.read_parquet(paths.DATA_PROCESSED / 'synthetic_v2_meta.parquet')
por_part = meta.groupby('partition')['session_id'].nunique()
solape = {}
for a in ('train', 'validation', 'test'):
    for b in ('train', 'validation', 'test'):
        if a < b:
            sa = set(meta.loc[meta.partition == a, 'session_id'])
            sb = set(meta.loc[meta.partition == b, 'session_id'])
            solape[f'{a}∩{b}'] = len(sa & sb)
print('sesiones por partición:', por_part.to_dict())
print('sesiones compartidas  :', solape)
assert all(v == 0 for v in solape.values()), 'hay fuga entre particiones'

sesiones por partición: {'test': 12, 'train': 48, 'validation': 12}
sesiones compartidas  : {'train∩validation': 0, 'test∩train': 0, 'test∩validation': 0}


## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
